In [ ]:
import os
import gc
import pickle
import pandas as pd
from tqdm import tqdm

from torch_geometric.loader import DataLoader
from data import ProteinECNumGraphDataset
from egnn_model import GraphEC
import torch.nn as nn
import torch

/home/liangpu/miniforge3/envs/graphec/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda:0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

config = {
    'node_input_dim': 9+184,  # 21 + 184, # precomputed + updated
    'edge_input_dim': 450,
    'hidden_dim': 512,
    'layer': 1,
    # 'augment_eps': 0.15,
    'augment_eps': 0,
    'class_align_weight': 0.2,
    'batch_size': 4,
    'folds': 5,
    'r': 16,
    'num_workers': 8,
    "random_seed": 0
}
torch.manual_seed(config["random_seed"])

In [ ]:
with open("./data_index/ec_map.pkl", "rb") as f:
    ec_map = pickle.load(f)

with open("./data_index/all_ec_num.pkl", "rb") as f:
    all_ec_num_df = pickle.load(f)

In [ ]:
with open("./data_index/testing_set.pkl", "rb") as f:
    test_data = pickle.load(f)

test_data = ProteinECNumGraphDataset(test_data, radius=config['r'])
test_dataloader = DataLoader(
    test_data,
    batch_size=config['batch_size'],
    shuffle=False,
    drop_last=False,
    num_workers=config['num_workers'],
    prefetch_factor=2
)

In [ ]:
model = torch.load("test_model.pt")
model = model.eval().to(device)

In [6]:
def split_result(output):
    ec4_pred = torch.sort(torch.sigmoid(output), descending=True)

    ec3_map = [(
        sum(ec_map[2][:i]), sum(ec_map[2][:i+1])
    )for i in range(len(ec_map[2]))]
    ec3_output = torch.stack([output[:, p1:p2].max(axis=1).values for p1, p2 in ec3_map]).T
    ec3_pred = torch.sort(torch.sigmoid(ec3_output), descending=True)

    ec2_map = [(
        sum(ec_map[1][:i]), sum(ec_map[1][:i+1])
    )for i in range(len(ec_map[1]))]
    ec2_output = torch.stack([ec3_output[:, p1:p2].max(axis=1).values for p1, p2 in ec2_map]).T
    ec2_pred = torch.sort(torch.sigmoid(ec2_output), descending=True)

    ec1_map = [(
        sum(ec_map[0][:i]), sum(ec_map[0][:i+1])
    )for i in range(len(ec_map[0]))]
    ec1_output = torch.stack([ec2_output[:, p1:p2].max(axis=1).values for p1, p2 in ec1_map]).T
    ec1_pred = torch.sort(torch.sigmoid(ec1_output), descending=True)
    return ec4_pred, ec3_pred, ec2_pred, ec1_pred

In [7]:
def correct_count(pred, true, batch_size, top_n=1):
    true = true.reshape(batch_size, -1).argwhere()
    all_label = true.shape[0]
    correct_label = 0
    for batch_idx, label in true:
        label_count = len([i for i in true[:, 0] if i==batch_idx])
        # if(label in pred.indices[batch_idx][:top_n]):
        if(label in pred.indices[batch_idx][:label_count]):
            correct_label+=1
    return correct_label, all_label

In [8]:
all_ec_names = [   
    f"{r.ec1}.{r.ec2}.{r.ec3}.{r.ec4}"
    for _,r in all_ec_num_df.iterrows()
]

In [ ]:
total_class_count = [0,0,0,0]
correct_class_count = [0,0,0,0]
all_pred_ecnum = []
all_name = []

for data in tqdm(test_dataloader):
    data = data.to(device)
    with torch.no_grad():
        output = model.forward(
            data.X, data.structure_feat, data.seq_feat,
            data.edge_index, data.batch
        ).to("cpu")
    data = data.to("cpu")
    batch_size = output.shape[0]
    pred = split_result(output)
    true = (data.ec4_label, data.ec3_label, data.ec2_label, data.ec1_label)
    for i, (p, t) in enumerate(zip(pred, true)):
        corr_num, total_num = correct_count(p, t, batch_size, top_n=1)
        total_class_count[i] += total_num
        correct_class_count[i] += corr_num
    # Top-1
    all_pred_ecnum.extend([all_ec_names[i] for i in output.argmax(dim=1)])
    all_name.extend(data.name)
    gc.collect()

total_class_count = [i for i in reversed(total_class_count)]
correct_class_count = [i for i in reversed(correct_class_count)]

100%|██████████| 572/572 [02:27<00:00,  3.88it/s]


In [ ]:
res = pd.DataFrame({"id":all_name,"pred":all_pred_ecnum})
res.to_csv("test_top1.csv",index=False)
res

,id,pred
0,Q9I0R8,2.7.13.3
1,Q3J189,1.14.14.69
2,P0DW53,4.2.3.136
3,V9TS54,3.2.1.8
4,B6QK76,2.4.1.159
...,...,...
2281,S8AWN7,4.2.1.158
2282,P0DXE7,2.7.11.1
2283,C1DMX5,2.4.1.273
2284,F1SVF6,4.2.1.30


In [10]:
for c,t in zip(correct_class_count, total_class_count):
    print(c/t, c, t)

0.8134802882577363 1919 2359
0.7331524197195839 1621 2211
0.6913521261347348 1447 2093
0.5387931034482759 500 928


In [ ]:
""" 
Ori:
    0.8134802882577363 1919 2359
    0.7331524197195839 1621 2211
    0.6913521261347348 1447 2093
    0.5398706896551724 501 928

No Structure(graph):
    0.7939805002119542 1873 2359
    0.7060153776571687 1561 2211
    0.6751075011944577 1413 2093
    0.5926724137931034 550 928

No Sequence:
    0.7901653242899533 1864 2359
    0.6965174129353234 1540 2211
    0.6660296225513617 1394 2093
    0.5258620689655172 488 928

No Attn:
    0.4841034336583298 1142 2359
    0.24830393487109906 549 2211
    0.2264691829909221 474 2093
    0.2349137931034483 218 928
"""